# Module 6 • Transformers

# Lesson 32 • Transformer Encoder Models and Contextual Embeddings

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Estimated study time:** 170–210 minutes  
**Execution target:** CPU

---

## Scope

This lesson introduces encoder-only Transformer models and contextual
embeddings. It explains bidirectional context, masked language modeling,
special tokens, dynamic masking, contextual token representations, sentence
pooling, transfer learning, frozen-feature evaluation, and downstream
fine-tuning.

The notebook trains a compact Transformer encoder on a synthetic masked
language-modeling corpus and reuses it for text classification. No pretrained
model download, internet connection, or GPU is required.

## Learning Objectives

After completing this lesson, the learner should be able to:

- distinguish static and contextual embeddings;
- explain encoder-only Transformer models;
- describe bidirectional contextualization;
- use PAD, UNK, CLS, SEP, and MASK tokens;
- explain masked language modeling;
- construct masked training examples;
- implement an encoder-only Transformer;
- train a compact masked language model;
- extract contextual token embeddings;
- compare the same word across different contexts;
- create sentence embeddings with CLS and mean pooling;
- transfer a pretrained encoder to classification;
- compare frozen and fine-tuned encoders;
- evaluate predictions and errors;
- discuss subword tokenization and Arabic morphology.

## Table of Contents

1. Encoder-Only Transformer Models
2. Static Versus Contextual Embeddings
3. Bidirectional Context
4. Special Tokens
5. Masked Language Modeling
6. MLM Corruption Strategy
7. Contextual Representation Shapes
8. Synthetic Pretraining Corpus
9. Tokenization
10. Vocabulary Construction
11. Sequence Encoding
12. MLM Dataset
13. Dynamic Padding
14. Positional Encoding
15. Transformer Encoder
16. Masked-Language-Model Head
17. Complete Encoder Model
18. Model Shape Inspection
19. MLM Loss
20. MLM Training Loop
21. MLM Learning Curves
22. Mask Prediction
23. Contextual Token Embeddings
24. Polysemy: The Word “Bank”
25. Contextual Similarity
26. Sentence Pooling
27. CLS Pooling
28. Mean Pooling
29. Downstream Classification Dataset
30. Classification Splits
31. Classification Dataset and Batching
32. Transfer-Learning Classifier
33. Frozen Encoder Training
34. Fine-Tuned Encoder Training
35. Model Comparison
36. Test Evaluation
37. Confusion Matrix
38. Error Analysis
39. Catastrophic Forgetting
40. Domain Shift
41. Tokenization and Subwords
42. Arabic and Multilingual Considerations
43. Reproducibility and Reporting
44. Knowledge Check
45. Exercises
46. Summary and Next Lesson

# 1. Encoder-Only Transformer Models

Encoder-only Transformers process the complete input sequence and produce one
contextual representation per token.

Common applications:

- text classification;
- sequence labeling;
- extractive question answering;
- semantic similarity;
- retrieval;
- token-level feature extraction.

In [ ]:
import copy
import math
import random
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch import nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, Dataset

encoder_tasks = pd.DataFrame(
    [
        ("Text classification", "CLS or pooled representation"),
        ("NER", "one contextual vector per token"),
        ("Similarity", "sentence embeddings"),
        ("Retrieval", "query and document embeddings"),
        ("Question answering", "token-span prediction"),
    ],
    columns=["Task", "Encoder output used"],
)

encoder_tasks

Encoder-only models do not generate target sequences autoregressively.

# 2. Static Versus Contextual Embeddings

A static embedding assigns one vector to each vocabulary item.

A contextual model assigns a vector based on both the token and its surrounding
sequence.

In [ ]:
embedding_comparison = pd.DataFrame(
    [
        (
            "Static",
            "one vector per token type",
            "bank has one representation",
        ),
        (
            "Contextual",
            "one vector per token occurrence",
            "bank changes by context",
        ),
    ],
    columns=["Embedding type", "Representation rule", "Polysemy"],
)

embedding_comparison

# 3. Bidirectional Context

Encoder self-attention can use tokens on both sides of a position.

```text
left context ← token → right context
```

This is suitable when the full input is available.

Bidirectional context differs from causal language modeling, where future
tokens are hidden.

# 4. Special Tokens

This notebook uses:

- `<PAD>` for batch padding;
- `<UNK>` for unknown words;
- `<CLS>` for sequence-level representation;
- `<SEP>` for sequence termination or separation;
- `<MASK>` for masked language modeling.

In [ ]:
special_token_table = pd.DataFrame(
    [
        ("<PAD>", "padding", "ignored by attention and loss"),
        ("<UNK>", "unknown token", "OOV fallback"),
        ("<CLS>", "classification token", "sequence representation"),
        ("<SEP>", "separator", "marks sequence boundary"),
        ("<MASK>", "masked token", "MLM prediction target"),
    ],
    columns=["Token", "Purpose", "Typical use"],
)

special_token_table

# 5. Masked Language Modeling

Masked language modeling hides selected tokens and asks the encoder to predict
the originals.

Example:

```text
input:  the customer visited the <MASK>
target: bank
```

MLM trains bidirectional contextual representations because prediction can use
both left and right context.

# 6. MLM Corruption Strategy

A standard conceptual strategy selects approximately 15% of tokens.

Selected tokens may be:

- replaced with MASK;
- replaced with a random token;
- left unchanged.

Only selected positions contribute to MLM loss.

In [ ]:
corruption_summary = pd.DataFrame(
    [
        ("MASK replacement", 0.80),
        ("Random token", 0.10),
        ("Unchanged", 0.10),
    ],
    columns=["Selected-token action", "Typical probability"],
)

corruption_summary

The exact percentages are design choices, not universal laws.

# 7. Contextual Representation Shapes

For batch size `B`, sequence length `T`, and hidden dimension `D`:

```text
token IDs:               (B, T)
contextual embeddings:   (B, T, D)
MLM logits:              (B, T, vocabulary_size)
CLS representation:      (B, D)
```

# 8. Synthetic Pretraining Corpus

The corpus contains finance, river, health, technology, and travel contexts.
The word `bank` appears with two senses.

In [ ]:
pretraining_sentences = [
    "the customer deposited money at the bank",
    "the bank approved the customer loan",
    "the bank reviewed the financial account",
    "the customer visited the bank for payment",
    "the bank transferred money safely",
    "the loan officer works at the bank",
    "the account belongs to the customer",
    "the payment reached the bank today",
    "the river bank was covered with green grass",
    "the child sat beside the river bank",
    "the boat moved near the river bank",
    "birds rested along the quiet river bank",
    "the river bank became muddy after rain",
    "trees grow beside the river bank",
    "the doctor treated the patient",
    "the nurse provided medicine at the clinic",
    "the hospital scheduled a medical examination",
    "exercise supports long term health",
    "the patient recovered after treatment",
    "the clinic reviewed the diagnosis",
    "the software update caused a network error",
    "the application connected to the server",
    "the computer stored data safely",
    "the network upload failed today",
    "the device installed the software update",
    "the server restarted after the error",
    "the tourist booked a hotel near the airport",
    "the flight arrived after a short delay",
    "the passenger collected the luggage",
    "the travel ticket changed today",
    "the tourist visited the city museum",
    "the hotel confirmed the reservation",
]

pretraining_frame = pd.DataFrame(
    {
        "text": (
            pretraining_sentences
            * 5
        )
    }
)

print(
    "Pretraining sequences:",
    len(pretraining_frame),
)

Repetition increases the number of small training examples while keeping the
notebook CPU-friendly.

# 9. Tokenization

In [ ]:
TOKEN_PATTERN = re.compile(
    r"\b\w+(?:[-']\w+)*\b",
    flags=re.UNICODE,
)


def tokenize(
    text: str,
) -> list[str]:
    return TOKEN_PATTERN.findall(
        text.lower()
    )


tokenize(
    "The bank approved the customer's loan."
)

# 10. Vocabulary Construction

In [ ]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
CLS_TOKEN = "<CLS>"
SEP_TOKEN = "<SEP>"
MASK_TOKEN = "<MASK>"

SPECIAL_TOKENS = [
    PAD_TOKEN,
    UNK_TOKEN,
    CLS_TOKEN,
    SEP_TOKEN,
    MASK_TOKEN,
]

token_counts = Counter(
    token
    for text in pretraining_frame[
        "text"
    ]
    for token in tokenize(text)
)

vocabulary = (
    SPECIAL_TOKENS
    + sorted(token_counts)
)

token_to_index = {
    token: index
    for index, token
    in enumerate(vocabulary)
}

PAD_ID = token_to_index[
    PAD_TOKEN
]
UNK_ID = token_to_index[
    UNK_TOKEN
]
CLS_ID = token_to_index[
    CLS_TOKEN
]
SEP_ID = token_to_index[
    SEP_TOKEN
]
MASK_ID = token_to_index[
    MASK_TOKEN
]

print(
    "Vocabulary size:",
    len(vocabulary),
)

# 11. Sequence Encoding

In [ ]:
MAX_LENGTH = 14


def encode_text(
    text: str,
    maximum_length: int = MAX_LENGTH,
) -> list[int]:
    content = [
        token_to_index.get(
            token,
            UNK_ID,
        )
        for token in tokenize(text)
    ]

    token_ids = (
        [CLS_ID]
        + content[
            :maximum_length - 2
        ]
        + [SEP_ID]
    )

    return token_ids


example_ids = encode_text(
    "the bank approved the loan"
)

print(example_ids)
print(
    [
        vocabulary[token_id]
        for token_id in example_ids
    ]
)

# 12. MLM Dataset

Masking is created deterministically during dataset construction so notebook
execution is reproducible.

In [ ]:
def create_masked_example(
    token_ids: list[int],
    generator: random.Random,
    selection_probability: float = 0.20,
) -> tuple[
    list[int],
    list[int],
]:
    corrupted = list(token_ids)
    labels = [-100] * len(token_ids)

    candidate_positions = [
        position
        for position, token_id
        in enumerate(token_ids)
        if token_id not in {
            CLS_ID,
            SEP_ID,
            PAD_ID,
        }
    ]

    selected = [
        position
        for position
        in candidate_positions
        if (
            generator.random()
            < selection_probability
        )
    ]

    if (
        not selected
        and candidate_positions
    ):
        selected = [
            generator.choice(
                candidate_positions
            )
        ]

    ordinary_token_ids = list(
        range(
            len(SPECIAL_TOKENS),
            len(vocabulary),
        )
    )

    for position in selected:
        original_id = token_ids[
            position
        ]
        labels[position] = (
            original_id
        )

        action = generator.random()

        if action < 0.80:
            corrupted[position] = (
                MASK_ID
            )
        elif action < 0.90:
            corrupted[position] = (
                generator.choice(
                    ordinary_token_ids
                )
            )
        else:
            corrupted[position] = (
                original_id
            )

    return corrupted, labels


masked_ids, masked_labels = (
    create_masked_example(
        example_ids,
        random.Random(42),
    )
)

print(
    [
        vocabulary[token_id]
        for token_id in masked_ids
    ]
)
print(masked_labels)

In [ ]:
class MLMDataset(Dataset):
    def __init__(
        self,
        texts,
        seed: int = 42,
    ):
        generator = random.Random(
            seed
        )

        self.examples = []

        for text in texts:
            token_ids = encode_text(
                text
            )

            corrupted, labels = (
                create_masked_example(
                    token_ids,
                    generator,
                )
            )

            self.examples.append(
                {
                    "input_ids": torch.tensor(
                        corrupted,
                        dtype=torch.long,
                    ),
                    "labels": torch.tensor(
                        labels,
                        dtype=torch.long,
                    ),
                    "text": text,
                }
            )

    def __len__(self):
        return len(
            self.examples
        )

    def __getitem__(
        self,
        index,
    ):
        return self.examples[
            index
        ]


mlm_dataset = MLMDataset(
    pretraining_frame[
        "text"
    ],
    seed=42,
)

len(mlm_dataset)

# 13. Dynamic Padding

In [ ]:
def collate_mlm_batch(
    batch,
):
    maximum = max(
        len(item["input_ids"])
        for item in batch
    )

    input_ids = torch.full(
        (
            len(batch),
            maximum,
        ),
        PAD_ID,
        dtype=torch.long,
    )

    labels = torch.full(
        (
            len(batch),
            maximum,
        ),
        -100,
        dtype=torch.long,
    )

    for row, item in enumerate(
        batch
    ):
        length = len(
            item["input_ids"]
        )

        input_ids[
            row,
            :length,
        ] = item[
            "input_ids"
        ]

        labels[
            row,
            :length,
        ] = item[
            "labels"
        ]

    return {
        "input_ids": input_ids,
        "padding_mask": (
            input_ids == PAD_ID
        ),
        "labels": labels,
        "texts": [
            item["text"]
            for item in batch
        ],
    }


mlm_loader = DataLoader(
    mlm_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_mlm_batch,
    generator=torch.Generator().manual_seed(
        42
    ),
)

sample_mlm_batch = next(
    iter(mlm_loader)
)

print(
    sample_mlm_batch[
        "input_ids"
    ].shape
)

# 14. Positional Encoding

In [ ]:
class SinusoidalPositionalEncoding(
    nn.Module
):
    def __init__(
        self,
        model_dimension: int,
        maximum_length: int = 128,
    ):
        super().__init__()

        encoding = torch.zeros(
            maximum_length,
            model_dimension,
        )

        positions = torch.arange(
            maximum_length,
            dtype=torch.float32,
        ).unsqueeze(1)

        rates = torch.exp(
            torch.arange(
                0,
                model_dimension,
                2,
                dtype=torch.float32,
            )
            * (
                -math.log(10000.0)
                / model_dimension
            )
        )

        encoding[
            :,
            0::2,
        ] = torch.sin(
            positions * rates
        )

        encoding[
            :,
            1::2,
        ] = torch.cos(
            positions * rates
        )

        self.register_buffer(
            "encoding",
            encoding.unsqueeze(0),
        )

    def forward(
        self,
        embeddings: torch.Tensor,
    ) -> torch.Tensor:
        return (
            embeddings
            + self.encoding[
                :,
                :embeddings.size(1),
                :,
            ]
        )

# 15. Transformer Encoder

In [ ]:
class TinyTransformerEncoder(
    nn.Module
):
    def __init__(
        self,
        vocabulary_size: int,
        model_dimension: int = 32,
        head_count: int = 4,
        feed_forward_dimension: int = 64,
        layer_count: int = 2,
        dropout: float = 0.10,
    ):
        super().__init__()

        self.model_dimension = (
            model_dimension
        )

        self.embedding = nn.Embedding(
            vocabulary_size,
            model_dimension,
            padding_idx=PAD_ID,
        )

        self.position = (
            SinusoidalPositionalEncoding(
                model_dimension,
                maximum_length=MAX_LENGTH,
            )
        )

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=model_dimension,
                nhead=head_count,
                dim_feedforward=(
                    feed_forward_dimension
                ),
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
        )

        self.encoder = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=layer_count,
            )
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        input_ids: torch.Tensor,
        padding_mask: torch.Tensor,
    ) -> torch.Tensor:
        embeddings = (
            self.embedding(
                input_ids
            )
            * math.sqrt(
                self.model_dimension
            )
        )

        embeddings = self.position(
            embeddings
        )

        return self.encoder(
            self.dropout(
                embeddings
            ),
            src_key_padding_mask=(
                padding_mask
            ),
        )

# 16. Masked-Language-Model Head

In [ ]:
class MLMHead(nn.Module):
    def __init__(
        self,
        model_dimension: int,
        vocabulary_size: int,
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                model_dimension,
                model_dimension,
            ),
            nn.GELU(),
            nn.LayerNorm(
                model_dimension
            ),
            nn.Linear(
                model_dimension,
                vocabulary_size,
            ),
        )

    def forward(
        self,
        contextual_states,
    ):
        return self.network(
            contextual_states
        )

# 17. Complete Encoder Model

In [ ]:
class TinyMaskedLanguageModel(
    nn.Module
):
    def __init__(
        self,
        vocabulary_size: int,
        model_dimension: int = 32,
    ):
        super().__init__()

        self.encoder = (
            TinyTransformerEncoder(
                vocabulary_size=(
                    vocabulary_size
                ),
                model_dimension=(
                    model_dimension
                ),
            )
        )

        self.mlm_head = MLMHead(
            model_dimension=(
                model_dimension
            ),
            vocabulary_size=(
                vocabulary_size
            ),
        )

    def forward(
        self,
        input_ids: torch.Tensor,
        padding_mask: torch.Tensor,
    ):
        contextual_states = (
            self.encoder(
                input_ids,
                padding_mask,
            )
        )

        logits = self.mlm_head(
            contextual_states
        )

        return {
            "logits": logits,
            "contextual_states": (
                contextual_states
            ),
        }


DEVICE = torch.device(
    "cpu"
)

torch.manual_seed(42)

mlm_model = (
    TinyMaskedLanguageModel(
        vocabulary_size=len(
            vocabulary
        ),
        model_dimension=32,
    ).to(DEVICE)
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter
        in mlm_model.parameters()
    ),
)

# 18. Model Shape Inspection

In [ ]:
with torch.no_grad():
    shape_output = mlm_model(
        sample_mlm_batch[
            "input_ids"
        ].to(DEVICE),
        sample_mlm_batch[
            "padding_mask"
        ].to(DEVICE),
    )

print(
    "Contextual states:",
    shape_output[
        "contextual_states"
    ].shape,
)
print(
    "MLM logits:",
    shape_output[
        "logits"
    ].shape,
)

# 19. MLM Loss

Unselected positions use label `-100`, which PyTorch cross-entropy ignores.

In [ ]:
mlm_loss_function = (
    nn.CrossEntropyLoss(
        ignore_index=-100
    )
)


def calculate_mlm_loss(
    logits: torch.Tensor,
    labels: torch.Tensor,
) -> torch.Tensor:
    return mlm_loss_function(
        logits.reshape(
            -1,
            logits.size(-1),
        ),
        labels.reshape(-1),
    )


initial_mlm_loss = (
    calculate_mlm_loss(
        shape_output["logits"],
        sample_mlm_batch[
            "labels"
        ].to(DEVICE),
    )
)

print(
    "Initial MLM loss:",
    float(initial_mlm_loss),
)

# 20. MLM Training Loop

In [ ]:
def set_seed(
    seed: int = 42,
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def train_mlm(
    model: nn.Module,
    loader: DataLoader,
    epochs: int = 28,
    learning_rate: float = 0.003,
):
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=1e-4,
    )

    history = []

    for epoch in range(epochs):
        model.train()

        losses = []
        accuracies = []
        gradient_norms = []

        for batch in loader:
            input_ids = batch[
                "input_ids"
            ].to(DEVICE)

            padding_mask = batch[
                "padding_mask"
            ].to(DEVICE)

            labels = batch[
                "labels"
            ].to(DEVICE)

            optimizer.zero_grad()

            output = model(
                input_ids,
                padding_mask,
            )

            loss = calculate_mlm_loss(
                output["logits"],
                labels,
            )

            loss.backward()

            gradient_norm = (
                clip_grad_norm_(
                    model.parameters(),
                    max_norm=5.0,
                )
            )

            optimizer.step()

            selected = (
                labels != -100
            )

            predictions = (
                output["logits"].argmax(
                    dim=-1
                )
            )

            accuracy = (
                (
                    predictions[
                        selected
                    ]
                    == labels[
                        selected
                    ]
                )
                .float()
                .mean()
                .item()
            )

            losses.append(
                float(loss.item())
            )
            accuracies.append(
                float(accuracy)
            )
            gradient_norms.append(
                float(gradient_norm)
            )

        history.append(
            {
                "epoch": epoch,
                "mlm_loss": float(
                    np.mean(losses)
                ),
                "masked_accuracy": float(
                    np.mean(accuracies)
                ),
                "gradient_norm": float(
                    np.mean(
                        gradient_norms
                    )
                ),
            }
        )

    return (
        model,
        pd.DataFrame(history),
    )


set_seed(42)

trained_mlm, mlm_history = (
    train_mlm(
        mlm_model,
        mlm_loader,
    )
)

print(
    "Final MLM loss:",
    round(
        mlm_history[
            "mlm_loss"
        ].iloc[-1],
        4,
    ),
)
print(
    "Final masked accuracy:",
    round(
        mlm_history[
            "masked_accuracy"
        ].iloc[-1],
        3,
    ),
)

# 21. MLM Learning Curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    mlm_history["epoch"],
    mlm_history["mlm_loss"],
)
plt.xlabel("Epoch")
plt.ylabel("MLM loss")
plt.title("Masked Language Modeling Loss")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    mlm_history["epoch"],
    mlm_history[
        "masked_accuracy"
    ],
)
plt.xlabel("Epoch")
plt.ylabel("Masked-token accuracy")
plt.title("Masked Language Modeling Accuracy")
plt.tight_layout()
plt.show()

# 22. Mask Prediction

In [ ]:
def predict_mask(
    model: TinyMaskedLanguageModel,
    text_with_mask: str,
    top_k: int = 5,
):
    tokens = tokenize(
        text_with_mask
    )

    token_ids = [
        CLS_ID
    ] + [
        (
            MASK_ID
            if token == "mask"
            else token_to_index.get(
                token,
                UNK_ID,
            )
        )
        for token in tokens
    ] + [
        SEP_ID
    ]

    input_ids = torch.tensor(
        [token_ids],
        dtype=torch.long,
        device=DEVICE,
    )

    padding_mask = torch.zeros_like(
        input_ids,
        dtype=torch.bool,
    )

    model.eval()

    with torch.no_grad():
        output = model(
            input_ids,
            padding_mask,
        )

        mask_position = token_ids.index(
            MASK_ID
        )

        probabilities = torch.softmax(
            output["logits"][
                0,
                mask_position,
            ],
            dim=0,
        )

        values, indices = torch.topk(
            probabilities,
            k=top_k,
        )

    return pd.DataFrame(
        {
            "token": [
                vocabulary[index]
                for index
                in indices.cpu().tolist()
            ],
            "probability": (
                values.cpu().tolist()
            ),
        }
    )


predict_mask(
    trained_mlm,
    "the customer visited the mask",
)

The tiny corpus limits prediction quality. The purpose is to expose the full
pretraining workflow.

# 23. Contextual Token Embeddings

In [ ]:
def contextualize(
    model: TinyMaskedLanguageModel,
    text: str,
):
    token_ids = encode_text(
        text
    )

    input_ids = torch.tensor(
        [token_ids],
        dtype=torch.long,
        device=DEVICE,
    )

    padding_mask = torch.zeros_like(
        input_ids,
        dtype=torch.bool,
    )

    model.eval()

    with torch.no_grad():
        states = model.encoder(
            input_ids,
            padding_mask,
        )[0].cpu().numpy()

    tokens = [
        vocabulary[token_id]
        for token_id in token_ids
    ]

    return tokens, states


tokens, states = contextualize(
    trained_mlm,
    "the bank approved the loan"
)

print(tokens)
print(states.shape)

# 24. Polysemy: The Word “Bank”

In [ ]:
finance_sentence = (
    "the bank approved the loan"
)

river_sentence = (
    "the child sat beside the river bank"
)

finance_tokens, finance_states = (
    contextualize(
        trained_mlm,
        finance_sentence,
    )
)

river_tokens, river_states = (
    contextualize(
        trained_mlm,
        river_sentence,
    )
)

finance_bank_position = (
    finance_tokens.index("bank")
)

river_bank_position = (
    river_tokens.index("bank")
)

finance_bank_vector = (
    finance_states[
        finance_bank_position
    ]
)

river_bank_vector = (
    river_states[
        river_bank_position
    ]
)

print(
    "Finance bank position:",
    finance_bank_position,
)
print(
    "River bank position:",
    river_bank_position,
)

# 25. Contextual Similarity

In [ ]:
def cosine_similarity(
    left: np.ndarray,
    right: np.ndarray,
) -> float:
    denominator = (
        np.linalg.norm(left)
        * np.linalg.norm(right)
    )

    return float(
        np.dot(left, right)
        / max(
            denominator,
            1e-12,
        )
    )


same_sense_tokens, same_sense_states = (
    contextualize(
        trained_mlm,
        "the customer visited the bank for payment",
    )
)

same_sense_bank = same_sense_states[
    same_sense_tokens.index("bank")
]

similarity_frame = pd.DataFrame(
    [
        (
            "finance vs finance",
            cosine_similarity(
                finance_bank_vector,
                same_sense_bank,
            ),
        ),
        (
            "finance vs river",
            cosine_similarity(
                finance_bank_vector,
                river_bank_vector,
            ),
        ),
    ],
    columns=[
        "Comparison",
        "Cosine similarity",
    ],
)

similarity_frame

Contextual similarity is model- and layer-dependent. This tiny model should be
treated as a demonstration rather than a semantic benchmark.

# 26. Sentence Pooling

Common pooling methods:

- CLS token;
- masked mean;
- masked maximum;
- learned pooling.

In [ ]:
pooling_summary = pd.DataFrame(
    [
        ("CLS", "dedicated first token"),
        ("Mean", "average valid token states"),
        ("Max", "largest value per dimension"),
        ("Learned", "task-trained weighting"),
    ],
    columns=["Pooling", "Mechanism"],
)

pooling_summary

# 27. CLS Pooling

In [ ]:
def cls_pool(
    contextual_states: torch.Tensor,
) -> torch.Tensor:
    return contextual_states[
        :,
        0,
        :,
    ]

# 28. Mean Pooling

In [ ]:
def masked_mean_pool(
    contextual_states: torch.Tensor,
    padding_mask: torch.Tensor,
) -> torch.Tensor:
    valid = (
        ~padding_mask
    ).unsqueeze(-1)

    summed = (
        contextual_states
        * valid
    ).sum(dim=1)

    counts = valid.sum(
        dim=1
    ).clamp(min=1)

    return summed / counts

# 29. Downstream Classification Dataset

In [ ]:
classification_records = [
    ("doctor treats patient in hospital", "health"),
    ("nurse provides medicine to patient", "health"),
    ("patient visits clinic for diagnosis", "health"),
    ("hospital schedules medical treatment", "health"),
    ("exercise supports long term health", "health"),
    ("nutrition improves patient recovery", "health"),
    ("doctor reviews the medical report", "health"),
    ("clinic provides emergency service", "health"),
    ("nurse helps the patient today", "health"),
    ("medicine reduces the health problem", "health"),
    ("hospital needs experienced doctors", "health"),
    ("patient requests treatment information", "health"),

    ("bank approves customer loan", "finance"),
    ("invoice contains payment charge", "finance"),
    ("customer requests card refund", "finance"),
    ("billing account has a problem", "finance"),
    ("loan interest increased today", "finance"),
    ("bank transfers money safely", "finance"),
    ("payment failed on the card", "finance"),
    ("refund request remains pending", "finance"),
    ("invoice price is incorrect", "finance"),
    ("customer updates bank account", "finance"),
    ("billing service changed the charge", "finance"),
    ("loan payment needs approval", "finance"),

    ("software update caused an error", "technology"),
    ("application cannot reach the server", "technology"),
    ("network upload failed today", "technology"),
    ("computer needs a system update", "technology"),
    ("device cannot install the software", "technology"),
    ("server lost important data", "technology"),
    ("application displays a network error", "technology"),
    ("computer connects to the server", "technology"),
    ("upload request failed again", "technology"),
    ("system update needs technical help", "technology"),
    ("device reports a software problem", "technology"),
    ("network service is unavailable", "technology"),

    ("flight arrives at airport", "travel"),
    ("tourist books hotel reservation", "travel"),
    ("airport lost passenger luggage", "travel"),
    ("travel ticket changed today", "travel"),
    ("flight delay affects the journey", "travel"),
    ("hotel reservation needs an update", "travel"),
    ("tourist visits the city museum", "travel"),
    ("beach trip starts tomorrow", "travel"),
    ("airport changes the flight gate", "travel"),
    ("passenger requests travel information", "travel"),
    ("journey includes a hotel stay", "travel"),
    ("ticket service reports a delay", "travel"),
]

classification_frame = pd.DataFrame(
    classification_records,
    columns=["text", "label"],
)

classification_frame[
    "label"
].value_counts()

# 30. Classification Splits

In [ ]:
(
    X_train_full,
    X_test,
    y_train_full,
    y_test,
) = train_test_split(
    classification_frame[
        "text"
    ],
    classification_frame[
        "label"
    ],
    test_size=0.25,
    random_state=42,
    stratify=classification_frame[
        "label"
    ],
)

(
    X_train,
    X_validation,
    y_train,
    y_validation,
) = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=42,
    stratify=y_train_full,
)

label_encoder = LabelEncoder()
label_encoder.fit(y_train)

print(
    "Train:",
    len(X_train),
)
print(
    "Validation:",
    len(X_validation),
)
print(
    "Test:",
    len(X_test),
)

# 31. Classification Dataset and Batching

In [ ]:
class EncoderClassificationDataset(
    Dataset
):
    def __init__(
        self,
        texts,
        labels,
    ):
        self.texts = list(
            texts
        )
        self.labels = (
            label_encoder.transform(
                list(labels)
            )
        )

    def __len__(self):
        return len(
            self.texts
        )

    def __getitem__(
        self,
        index,
    ):
        return {
            "input_ids": torch.tensor(
                encode_text(
                    self.texts[index]
                ),
                dtype=torch.long,
            ),
            "label": torch.tensor(
                self.labels[index],
                dtype=torch.long,
            ),
            "text": self.texts[index],
        }


def collate_classification_batch(
    batch,
):
    maximum = max(
        len(item["input_ids"])
        for item in batch
    )

    input_ids = torch.full(
        (
            len(batch),
            maximum,
        ),
        PAD_ID,
        dtype=torch.long,
    )

    labels = []

    for row, item in enumerate(
        batch
    ):
        length = len(
            item["input_ids"]
        )

        input_ids[
            row,
            :length,
        ] = item[
            "input_ids"
        ]

        labels.append(
            item["label"]
        )

    return {
        "input_ids": input_ids,
        "padding_mask": (
            input_ids == PAD_ID
        ),
        "labels": torch.stack(
            labels
        ),
        "texts": [
            item["text"]
            for item in batch
        ],
    }

In [ ]:
train_classification_dataset = (
    EncoderClassificationDataset(
        X_train,
        y_train,
    )
)

validation_classification_dataset = (
    EncoderClassificationDataset(
        X_validation,
        y_validation,
    )
)

test_classification_dataset = (
    EncoderClassificationDataset(
        X_test,
        y_test,
    )
)

train_classification_loader = (
    DataLoader(
        train_classification_dataset,
        batch_size=8,
        shuffle=True,
        collate_fn=(
            collate_classification_batch
        ),
        generator=(
            torch.Generator()
            .manual_seed(42)
        ),
    )
)

validation_classification_loader = (
    DataLoader(
        validation_classification_dataset,
        batch_size=8,
        shuffle=False,
        collate_fn=(
            collate_classification_batch
        ),
    )
)

test_classification_loader = (
    DataLoader(
        test_classification_dataset,
        batch_size=8,
        shuffle=False,
        collate_fn=(
            collate_classification_batch
        ),
    )
)

# 32. Transfer-Learning Classifier

In [ ]:
class EncoderClassifier(nn.Module):
    def __init__(
        self,
        encoder: TinyTransformerEncoder,
        class_count: int,
        freeze_encoder: bool,
        pooling: str = "cls",
    ):
        super().__init__()

        self.encoder = encoder
        self.pooling = pooling

        if freeze_encoder:
            for parameter in (
                self.encoder.parameters()
            ):
                parameter.requires_grad = (
                    False
                )

        self.dropout = nn.Dropout(
            0.15
        )

        self.classifier = nn.Linear(
            encoder.model_dimension,
            class_count,
        )

    def forward(
        self,
        input_ids: torch.Tensor,
        padding_mask: torch.Tensor,
    ):
        contextual_states = (
            self.encoder(
                input_ids,
                padding_mask,
            )
        )

        if self.pooling == "cls":
            representation = (
                cls_pool(
                    contextual_states
                )
            )
        elif self.pooling == "mean":
            representation = (
                masked_mean_pool(
                    contextual_states,
                    padding_mask,
                )
            )
        else:
            raise ValueError(
                "Unknown pooling method"
            )

        logits = self.classifier(
            self.dropout(
                representation
            )
        )

        return {
            "logits": logits,
            "representation": (
                representation
            ),
        }

The pretrained encoder weights are copied so frozen and fine-tuned experiments
start from the same representation.

In [ ]:
pretrained_encoder_state = (
    copy.deepcopy(
        trained_mlm.encoder.state_dict()
    )
)


def create_classifier(
    freeze_encoder: bool,
) -> EncoderClassifier:
    encoder = TinyTransformerEncoder(
        vocabulary_size=len(
            vocabulary
        ),
        model_dimension=32,
    )

    encoder.load_state_dict(
        pretrained_encoder_state
    )

    return EncoderClassifier(
        encoder=encoder,
        class_count=len(
            label_encoder.classes_
        ),
        freeze_encoder=(
            freeze_encoder
        ),
        pooling="cls",
    ).to(DEVICE)

# 33. Frozen Encoder Training

In [ ]:
classification_loss = (
    nn.CrossEntropyLoss()
)


def evaluate_classifier(
    model: nn.Module,
    loader: DataLoader,
):
    model.eval()

    losses = []
    labels_all = []
    predictions_all = []
    probabilities_all = []
    texts_all = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch[
                "input_ids"
            ].to(DEVICE)

            padding_mask = batch[
                "padding_mask"
            ].to(DEVICE)

            labels = batch[
                "labels"
            ].to(DEVICE)

            output = model(
                input_ids,
                padding_mask,
            )

            loss = classification_loss(
                output["logits"],
                labels,
            )

            probabilities = (
                torch.softmax(
                    output["logits"],
                    dim=1,
                )
            )

            predictions = (
                probabilities.argmax(
                    dim=1
                )
            )

            losses.append(
                float(loss.item())
            )
            labels_all.extend(
                labels.cpu().tolist()
            )
            predictions_all.extend(
                predictions.cpu().tolist()
            )
            probabilities_all.extend(
                probabilities.cpu().tolist()
            )
            texts_all.extend(
                batch["texts"]
            )

    return {
        "loss": float(
            np.mean(losses)
        ),
        "accuracy": accuracy_score(
            labels_all,
            predictions_all,
        ),
        "macro_f1": f1_score(
            labels_all,
            predictions_all,
            average="macro",
        ),
        "labels": np.asarray(
            labels_all
        ),
        "predictions": np.asarray(
            predictions_all
        ),
        "probabilities": np.asarray(
            probabilities_all
        ),
        "texts": texts_all,
    }

In [ ]:
def train_classifier(
    model: nn.Module,
    epochs: int,
    learning_rate: float,
    patience: int = 8,
):
    trainable_parameters = [
        parameter
        for parameter
        in model.parameters()
        if parameter.requires_grad
    ]

    optimizer = torch.optim.Adam(
        trainable_parameters,
        lr=learning_rate,
        weight_decay=1e-4,
    )

    best_state = copy.deepcopy(
        model.state_dict()
    )
    best_validation_loss = float(
        "inf"
    )
    without_improvement = 0
    history = []

    for epoch in range(epochs):
        model.train()

        losses = []

        for batch in (
            train_classification_loader
        ):
            input_ids = batch[
                "input_ids"
            ].to(DEVICE)

            padding_mask = batch[
                "padding_mask"
            ].to(DEVICE)

            labels = batch[
                "labels"
            ].to(DEVICE)

            optimizer.zero_grad()

            output = model(
                input_ids,
                padding_mask,
            )

            loss = classification_loss(
                output["logits"],
                labels,
            )

            loss.backward()

            clip_grad_norm_(
                trainable_parameters,
                max_norm=5.0,
            )

            optimizer.step()

            losses.append(
                float(loss.item())
            )

        validation_metrics = (
            evaluate_classifier(
                model,
                validation_classification_loader,
            )
        )

        history.append(
            {
                "epoch": epoch,
                "training_loss": float(
                    np.mean(losses)
                ),
                "validation_loss": (
                    validation_metrics[
                        "loss"
                    ]
                ),
                "validation_accuracy": (
                    validation_metrics[
                        "accuracy"
                    ]
                ),
                "validation_macro_f1": (
                    validation_metrics[
                        "macro_f1"
                    ]
                ),
            }
        )

        if (
            validation_metrics[
                "loss"
            ]
            < best_validation_loss
            - 1e-5
        ):
            best_validation_loss = (
                validation_metrics[
                    "loss"
                ]
            )

            best_state = (
                copy.deepcopy(
                    model.state_dict()
                )
            )

            without_improvement = 0
        else:
            without_improvement += 1

        if (
            without_improvement
            >= patience
        ):
            break

    model.load_state_dict(
        best_state
    )

    return (
        model,
        pd.DataFrame(history),
    )


set_seed(42)

frozen_classifier = (
    create_classifier(
        freeze_encoder=True
    )
)

(
    trained_frozen_classifier,
    frozen_history,
) = train_classifier(
    frozen_classifier,
    epochs=35,
    learning_rate=0.01,
)

print(
    "Frozen best validation F1:",
    round(
        frozen_history[
            "validation_macro_f1"
        ].max(),
        3,
    ),
)

# 34. Fine-Tuned Encoder Training

In [ ]:
set_seed(42)

fine_tuned_classifier = (
    create_classifier(
        freeze_encoder=False
    )
)

(
    trained_fine_tuned_classifier,
    fine_tuned_history,
) = train_classifier(
    fine_tuned_classifier,
    epochs=35,
    learning_rate=0.002,
)

print(
    "Fine-tuned best validation F1:",
    round(
        fine_tuned_history[
            "validation_macro_f1"
        ].max(),
        3,
    ),
)

# 35. Model Comparison

In [ ]:
frozen_validation = (
    evaluate_classifier(
        trained_frozen_classifier,
        validation_classification_loader,
    )
)

fine_tuned_validation = (
    evaluate_classifier(
        trained_fine_tuned_classifier,
        validation_classification_loader,
    )
)

comparison_frame = pd.DataFrame(
    [
        (
            "Frozen encoder",
            frozen_validation[
                "accuracy"
            ],
            frozen_validation[
                "macro_f1"
            ],
        ),
        (
            "Fine-tuned encoder",
            fine_tuned_validation[
                "accuracy"
            ],
            fine_tuned_validation[
                "macro_f1"
            ],
        ),
    ],
    columns=[
        "Model",
        "Validation accuracy",
        "Validation macro F1",
    ],
)

comparison_frame

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    frozen_history["epoch"],
    frozen_history[
        "validation_loss"
    ],
    label="Frozen encoder",
)
plt.plot(
    fine_tuned_history["epoch"],
    fine_tuned_history[
        "validation_loss"
    ],
    label="Fine-tuned encoder",
)
plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("Transfer-Learning Validation Curves")
plt.legend()
plt.tight_layout()
plt.show()

# 36. Test Evaluation

The model is selected using validation macro F1.

In [ ]:
if (
    fine_tuned_validation[
        "macro_f1"
    ]
    >= frozen_validation[
        "macro_f1"
    ]
):
    selected_name = (
        "Fine-tuned encoder"
    )
    selected_model = (
        trained_fine_tuned_classifier
    )
else:
    selected_name = (
        "Frozen encoder"
    )
    selected_model = (
        trained_frozen_classifier
    )

test_metrics = (
    evaluate_classifier(
        selected_model,
        test_classification_loader,
    )
)

print(
    "Selected model:",
    selected_name,
)
print(
    "Test accuracy:",
    round(
        test_metrics[
            "accuracy"
        ],
        3,
    ),
)
print(
    "Test macro F1:",
    round(
        test_metrics[
            "macro_f1"
        ],
        3,
    ),
)

In [ ]:
actual_labels = (
    label_encoder.inverse_transform(
        test_metrics["labels"]
    )
)

predicted_labels = (
    label_encoder.inverse_transform(
        test_metrics[
            "predictions"
        ]
    )
)

print(
    classification_report(
        actual_labels,
        predicted_labels,
        zero_division=0,
    )
)

# 37. Confusion Matrix

In [ ]:
class_names = list(
    label_encoder.classes_
)

matrix = confusion_matrix(
    actual_labels,
    predicted_labels,
    labels=class_names,
)

pd.DataFrame(
    matrix,
    index=[
        f"actual_{label}"
        for label in class_names
    ],
    columns=[
        f"predicted_{label}"
        for label in class_names
    ],
)

# 38. Error Analysis

In [ ]:
error_frame = pd.DataFrame(
    {
        "text": test_metrics[
            "texts"
        ],
        "actual": actual_labels,
        "predicted": (
            predicted_labels
        ),
        "confidence": test_metrics[
            "probabilities"
        ].max(axis=1),
    }
)

error_frame["correct"] = (
    error_frame["actual"]
    == error_frame["predicted"]
)

error_frame.sort_values(
    ["correct", "confidence"],
    ascending=[True, True],
)

# 39. Catastrophic Forgetting

Fine-tuning changes pretrained representations. Aggressive updates can erase
useful general knowledge.

Mitigation strategies:

- smaller encoder learning rate;
- gradual unfreezing;
- fewer epochs;
- weight regularization;
- replay or multi-task objectives.

In [ ]:
forgetting_mitigation = pd.DataFrame(
    [
        ("Small learning rate", "limits encoder drift"),
        ("Gradual unfreezing", "adapts upper layers first"),
        ("Early stopping", "limits excessive specialization"),
        ("Multi-task loss", "retains pretraining behavior"),
    ],
    columns=["Method", "Purpose"],
)

forgetting_mitigation

# 40. Domain Shift

Pretraining and downstream data may differ in:

- vocabulary;
- topics;
- sentence length;
- style;
- language variety;
- annotation policy.

In [ ]:
domain_shift_signals = pd.DataFrame(
    [
        ("High UNK rate", "vocabulary mismatch"),
        ("Low frozen performance", "weak feature transfer"),
        ("Large fine-tuning gain", "task adaptation needed"),
        ("Validation instability", "small or shifted dataset"),
    ],
    columns=["Signal", "Interpretation"],
)

domain_shift_signals

# 41. Tokenization and Subwords

Real encoder models generally use subword tokenization.

Benefits:

- fewer unknown words;
- reusable word fragments;
- manageable vocabulary.

Costs:

- longer sequences;
- fragmented linguistic units;
- more complex token-to-word alignment.

# 42. Arabic and Multilingual Considerations

Arabic contextual encoders must address:

- attached clitics;
- rich inflection;
- optional tashkeel;
- orthographic variation;
- MSA and dialects;
- code-switching;
- uneven multilingual data.

In [ ]:
arabic_examples = pd.DataFrame(
    [
        (
            "وَسَيَكْتُبُونَهَا",
            "وَ + سَ + يَكْتُبُونَ + هَا",
        ),
        (
            "بِالْمَدْرَسَةِ",
            "بِ + الْمَدْرَسَةِ",
        ),
        (
            "كِتَابُهُمَا",
            "كِتَابُ + هُمَا",
        ),
    ],
    columns=[
        "Fully vocalized form",
        "Illustrative segmentation",
    ],
)

arabic_examples

A subword tokenizer may split one vocalized Arabic word into several pieces.
The resulting contextual representation depends on segmentation and pooling.

For tasks where tashkeel carries lexical or morphological information, it
should be preserved explicitly.

In [ ]:
arabic_design_choices = pd.DataFrame(
    [
        ("Preserve tashkeel", "retain vocalized distinctions"),
        ("Normalize selectively", "reduce controlled variation"),
        ("Use subwords", "improve coverage"),
        ("Use morphological segments", "expose clitic structure"),
        ("Track dialect", "avoid collapsing varieties"),
    ],
    columns=["Choice", "Purpose"],
)

arabic_design_choices

# 43. Reproducibility and Reporting

Report:

- pretraining corpus;
- tokenizer and vocabulary;
- masking probability and corruption strategy;
- maximum sequence length;
- model dimension;
- head count;
- encoder layers;
- feed-forward dimension;
- MLM epochs and optimizer;
- pooling method;
- frozen or fine-tuned encoder;
- downstream split;
- learning rates;
- random seeds;
- metrics;
- hardware.

In [ ]:
import platform

metadata = pd.Series(
    {
        "pretraining_sequences": len(
            pretraining_frame
        ),
        "vocabulary_size": len(
            vocabulary
        ),
        "maximum_length": MAX_LENGTH,
        "model_dimension": 32,
        "attention_heads": 4,
        "encoder_layers": 2,
        "mlm_epochs": len(
            mlm_history
        ),
        "selected_classifier": (
            selected_name
        ),
        "device": str(DEVICE),
        "random_seed": 42,
        "python_version": (
            platform.python_version()
        ),
        "numpy_version": (
            np.__version__
        ),
        "torch_version": (
            torch.__version__
        ),
    },
    name="Contextual encoder experiment",
)

metadata

# 44. Knowledge Check

1. What is an encoder-only Transformer?
2. How do contextual embeddings differ from static embeddings?
3. Why is MLM bidirectional?
4. What roles do CLS, SEP, and MASK play?
5. Which MLM positions contribute to loss?
6. Why are masked tokens sometimes left unchanged?
7. What shape do contextual token states have?
8. How can the same word receive different vectors?
9. How do CLS and mean pooling differ?
10. What is transfer learning?
11. How do frozen and fine-tuned encoders differ?
12. What is catastrophic forgetting?
13. What is domain shift?
14. Why are subwords common in encoder models?
15. How do Arabic clitics and tashkeel affect contextual encoding?

# 45. Exercises

## Exercise 1 — Masking

Compare masking probabilities of 10%, 15%, and 25%.

## Exercise 2 — Corruption Strategy

Compare MASK-only corruption with the 80/10/10 strategy.

## Exercise 3 — Layer Representations

Return representations from every encoder layer.

## Exercise 4 — Polysemy

Analyze contextual vectors for several ambiguous words.

## Exercise 5 — Pooling

Compare CLS, mean, and max pooling.

## Exercise 6 — Frozen Transfer

Train several classifier heads over a frozen encoder.

## Exercise 7 — Fine-Tuning

Compare encoder learning rates.

## Exercise 8 — Gradual Unfreezing

Unfreeze one Transformer layer at a time.

## Exercise 9 — Arabic MLM

Build a small fully vocalized Arabic MLM corpus.

## Exercise 10 — Subword Tokenization

Replace word tokens with a trained subword vocabulary.

## Challenge Exercises

1. Tie the MLM output projection to the input embedding table.
2. Add sentence-pair inputs with segment embeddings.
3. Add a warmup and decay learning-rate schedule.
4. Compare MLM pretraining against random initialization.
5. Implement token-classification fine-tuning.

# 46. Summary and Next Lesson

In this lesson:

- encoder-only Transformers produced contextual token representations;
- static and contextual embeddings were distinguished;
- bidirectional context was connected to masked language modeling;
- PAD, UNK, CLS, SEP, and MASK tokens were introduced;
- MLM corruption and selective loss were implemented;
- a compact Transformer encoder and MLM head were trained on CPU;
- masked-token predictions were inspected;
- the word `bank` was compared across financial and river contexts;
- CLS and masked-mean sentence pooling were implemented;
- the pretrained encoder was transferred to text classification;
- frozen and fine-tuned encoder strategies were compared;
- errors, catastrophic forgetting, and domain shift were analyzed;
- Arabic morphology, subword segmentation, and tashkeel were connected to
  contextual encoding.

## Next Lesson

**Lesson 33: Transformer Decoder Models and Autoregressive Language
Modeling** introduces causal self-attention, next-token prediction, decoding
temperature, top-k and nucleus sampling, perplexity, and controlled text
generation.

# References

- Devlin, J. et al. BERT and masked language modeling.
- Vaswani, A. et al. *Attention Is All You Need*.
- Liu, Y. et al. RoBERTa pretraining literature.
- Rogers, A., Kovaleva, O., & Rumshisky, A. BERT analysis literature.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.